# 00a — MAESTRO Western Classical: Data Preparation

**Thesis context:** This notebook selects and copies a representative subset of MIDI files from
the MAESTRO v3.0.0 dataset (Anderson et al., 2019) for use in baseline tokenisation and
fine-tuning experiments. MAESTRO contains piano recordings with aligned MIDI from the
International Piano-e-Competition, 2004–2018.

**Output:**
- `data/processed/western_classical/midi/` — 150 selected MIDI files
- `data/metadata/maestro_selected.csv` — per-file metadata (composer, title, year, duration)

**Sampling strategy:** Stratified by year to preserve the temporal distribution of the dataset.
Files are drawn from the pre-defined `train` split provided in the MAESTRO metadata CSV.


In [1]:
import sys
from pathlib import Path

# Locate project root regardless of where Jupyter was launched from.
# Searches upward for PROGRESS.md — the root marker.
_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
import pandas as pd
import shutil

RAW_DIR   = PROJECT_ROOT / "datasets" / "western_classical" / "maestro-v3.0.0"
OUT_MIDI  = PROJECT_ROOT / "data" / "processed" / "western_classical" / "midi"
META_DIR  = PROJECT_ROOT / "data" / "metadata"

OUT_MIDI.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and inspect MAESTRO metadata

In [3]:
df = pd.read_csv(RAW_DIR / "maestro-v3.0.0.csv")
print(f"Total records : {len(df)}")
print(f"Columns       : {df.columns.tolist()}")
print(f"Split counts  :\n{df['split'].value_counts()}")
print(f"Year range    : {df['year'].min()} – {df['year'].max()}")
df.head(3)

Total records : 1276
Columns       : ['canonical_composer', 'canonical_title', 'split', 'year', 'midi_filename', 'audio_filename', 'duration']
Split counts  :
split
train         962
test          177
validation    137
Name: count, dtype: int64
Year range    : 2004 – 2018


,canonical_composer,canonical_title,split,year,midi_filename,audio_filename,duration
0,Alban Berg,Sonata Op. 1,train,2018,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...,698.661160
1,Alban Berg,Sonata Op. 1,train,2008,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...,759.518471
2,Alban Berg,Sonata Op. 1,train,2017,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...,464.649433


In [4]:
# Duration stats (seconds)
print("Duration statistics (seconds):")
print(df[df['split'] == 'train']['duration'].describe().round(1))

Duration statistics (seconds):
count     962.0
mean      595.9
std       461.4
min        45.2
25%       272.3
50%       485.9
75%       709.0
max      2624.7
Name: duration, dtype: float64


## 2. Filter to training split

In [5]:
train_df = df[df['split'] == 'train'].copy()
print(f"Train-split records: {len(train_df)}")
print("\nFiles per year (train split):")
print(train_df['year'].value_counts().sort_index().to_string())

Train-split records: 962

Files per year (train split):
year
2004    103
2006     90
2008     99
2009     94
2011    128
2013     97
2014     80
2015     95
2017    106
2018     70


## 3. Stratified sample — 150 files, proportional across years

We sample proportionally so every competition year is represented in roughly its
original ratio. `random_state=42` makes the selection reproducible.


In [6]:
N_TARGET = 150

sample = (
    train_df
    .groupby("year", group_keys=False)
    .apply(lambda x: x.sample(
        min(len(x), max(1, round(N_TARGET * len(x) / len(train_df)))),
        random_state=42
    ))
)

# Rounding may produce slightly more or fewer than N_TARGET — trim / top-up
if len(sample) > N_TARGET:
    sample = sample.sample(N_TARGET, random_state=42)
elif len(sample) < N_TARGET:
    remaining = train_df[~train_df.index.isin(sample.index)]
    topup = remaining.sample(min(N_TARGET - len(sample), len(remaining)), random_state=42)
    sample = pd.concat([sample, topup])

print(f"Selected {len(sample)} files")
print("\nFiles per year in selection:")
print(sample['year'].value_counts().sort_index().to_string())

Selected 150 files

Files per year in selection:
year
2004    16
2006    14
2008    15
2009    15
2011    20
2013    15
2014    12
2015    15
2017    17
2018    11


/var/folders/88/d6f4kns57fz2gzfzhd0dktfw0000gn/T/ipykernel_73705/1737129423.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df


In [7]:
print("\nTop 15 composers in selection:")
print(sample['canonical_composer'].value_counts().head(15).to_string())


Top 15 composers in selection:
canonical_composer
Franz Schubert                  29
Frédéric Chopin                 23
Ludwig van Beethoven            18
Johann Sebastian Bach           17
Franz Liszt                     12
Alexander Scriabin               6
Joseph Haydn                     5
Domenico Scarlatti               5
Claude Debussy                   4
Wolfgang Amadeus Mozart          4
Robert Schumann                  3
Felix Mendelssohn                3
Alban Berg                       3
Franz Schubert / Franz Liszt     3
George Frideric Handel           2


## 4. Copy MIDI files to processed directory

In [8]:
copied_names = []
missing = []

for _, row in sample.iterrows():
    src = RAW_DIR / row['midi_filename']
    # Flatten the year/filename path into a single name separated by underscore
    flat_name = row['midi_filename'].replace('/', '_')
    dst = OUT_MIDI / flat_name
    if src.exists():
        shutil.copy2(src, dst)
        copied_names.append(flat_name)
    else:
        missing.append(str(src))
        copied_names.append(None)

print(f"Copied : {len([n for n in copied_names if n])} files")
if missing:
    print(f"Missing: {missing}")

Copied : 150 files


## 5. Save metadata and verify

In [9]:
sample = sample.copy()
sample['processed_filename'] = copied_names
sample.to_csv(META_DIR / "maestro_selected.csv", index=False)
print(f"Metadata saved → data/metadata/maestro_selected.csv")

# Verification
midi_files = list(OUT_MIDI.glob("*.midi")) + list(OUT_MIDI.glob("*.mid"))
print(f"\nMIDI files in output dir : {len(midi_files)}")
print(f"Duration range (min)     : {sample['duration'].min()/60:.1f} – {sample['duration'].max()/60:.1f}")
print(f"Total duration (hours)   : {sample['duration'].sum()/3600:.2f}")
print("\n✓ MAESTRO preparation complete.")

Metadata saved → data/metadata/maestro_selected.csv

MIDI files in output dir : 150
Duration range (min)     : 1.2 – 40.4
Total duration (hours)   : 24.75

✓ MAESTRO preparation complete.
